In [991]:
import torch
from torch_geometric.loader import NeighborLoader

class GraphDataLoader:
    def __init__(self, data, num_neighbors=[100, 100], batch_size=256, shuffle=True):
        """
        Initialise le DataLoader pour un graphe avec NeighborLoader.

        Args:
            data (Data): L'objet Data de torch_geometric.
            num_neighbors (list): Nombre de voisins à échantillonner à chaque saut.
            batch_size (int): Taille de batch.
            shuffle (bool): Si True, mélange les nœuds pour obtenir des batches différents à chaque époque.
        """
        self.data = data
        self.num_neighbors = num_neighbors
        self.batch_size = batch_size
        self.shuffle = shuffle

    def get_loader(self):
        """
        Retourne le NeighborLoader configuré pour l'échantillonnage.

        Returns:
            NeighborLoader: DataLoader pour l'entraînement avec l'échantillonnage de voisins.
        """

        loader = NeighborLoader(
            self.data,
            num_neighbors=self.num_neighbors,
            batch_size=self.batch_size,
            input_nodes=torch.arange(self.data.x.size(0), dtype=torch.long),  # Entiers pour input_nodes
            shuffle=self.shuffle
        )
        return loader

config = {
    'embedding_dim': 256,
    'input_drop': 0.2,
    'hidden_drop': 0.3,
    'feat_drop': 0.2,
    'embedding_shape1': 32,  # Nouvelle valeur
    'hidden_size': 11904,    # ## 2048 ----> 123008 ; 768 ----> 43648 ; 256 ----> 11904
    'num_entities': 2,
    'num_relations': 2,
    'label_smoothing': 0.1,
    'use_bias': True,
    'device':'cuda'
}


In [980]:
from torch.utils.data import Dataset, DataLoader

class TripletDataset(Dataset):
    """
    Custom dataset for positive and negative triplets to train the ConvE model.
    """
    def __init__(self, positive_triplets, negative_triplets, entity_embeddings, relation_embeddings):
        """
        Args:
            positive_triplets (torch.Tensor): Tensor of positive triplets (num_triplets, 3).
            negative_triplets (torch.Tensor): Tensor of negative triplets (num_triplets, 3).
            entity_embeddings (torch.Tensor): Embedding matrix for entities (num_entities, emb_dim).
            relation_embeddings (torch.Tensor): Embedding matrix for relations (num_relations, emb_dim).
        """
        self.positive_triplets = positive_triplets
        self.negative_triplets = negative_triplets
        self.entity_embeddings = entity_embeddings
        self.relation_embeddings = relation_embeddings
        self.triplets = torch.cat([positive_triplets, negative_triplets], dim=0)
        self.labels = torch.cat([torch.ones(len(positive_triplets)), torch.zeros(len(negative_triplets))], dim=0)

    def __len__(self):
        return len(self.triplets)

    def __getitem__(self, idx):
        triplet = self.triplets[idx]
        label = self.labels[idx]
        
        head_emb = self.entity_embeddings[triplet[0]]
        rel_emb = self.relation_embeddings[triplet[1]]
        tail_emb = self.entity_embeddings[triplet[2]]
        
        return head_emb, rel_emb, tail_emb, label

def create_data_loader(positive_triplets, negative_triplets, entity_embeddings, relation_embeddings, batch_size, shuffle = True):
    """
    Creates a DataLoader for the ConvE model.

    Args:
        positive_triplets (torch.Tensor): Tensor of positive triplets (num_triplets, 3).
        negative_triplets (torch.Tensor): Tensor of negative triplets (num_triplets, 3).
        entity_embeddings (torch.Tensor): Embedding matrix for entities (num_entities, emb_dim).
        relation_embeddings (torch.Tensor): Embedding matrix for relations (num_relations, emb_dim).
        batch_size (int): Batch size for the data loader.

    Returns:
        DataLoader: DataLoader for training the ConvE model.
    """
    dataset = TripletDataset(positive_triplets, negative_triplets, entity_embeddings, relation_embeddings)
    data_loader = DataLoader(dataset, batch_size=batch_size, shuffle = shuffle)
    return data_loader

def generate_relation_embeddings_tensor(relations, embedding_size,  device ,seed=42):
    """
    Génère un tenseur d'embedding pour des relations uniques.
    
    Args:
        relations (list): Liste des relations uniques.
        embedding_size (int): Taille des vecteurs d'embedding.
        seed (int): Seed pour garantir la reproductibilité.
        
    Returns:
        torch.Tensor: Tenseur contenant les embeddings de taille [nb_relations, embedding_size].
    """
    # Fixer le seed pour la reproductibilité
    torch.manual_seed(seed)
    
    # Nombre de relations uniques
    num_relations = len(relations)
    
    # Définir une couche d'embedding
    embedding_layer = torch.nn.Embedding(num_relations, embedding_size)
    
    # Générer les embeddings pour tous les indices
    indices = torch.arange(num_relations)
    embeddings = embedding_layer(indices)  # Shape: [num_relations, embedding_size]
    
    return embeddings.to(device)

In [992]:
import sys
sys.path.append("../../utilities")
sys.path.append("../../data")
import utilities as u
from GraphDataPreparation import GraphDataPreparation



Entities_path = "../outputs/EntitiesBertEmbeddingAugmented.pickle"
Edges = "../outputs/PredicatesBertEmbeddingAugmented.pickle"
KG_path = "../../data/graph_filtred_gpt_20.json"

gdp = GraphDataPreparation(Entities_path, KG_path,edges_embd_path = Edges,is_directed = True)
data = gdp.prepare_graph_with_type()


Building NetworkX graph with unique relation type IDs
----------- Unique relation -------> 110
Building PyTorch Geometric Data object with unique relation type IDs


In [982]:
from torch_geometric.loader import NeighborLoader

G1_data_loader = NeighborLoader(
    data,  # Le graphe unique
    num_neighbors = [214, 150],  # Nombre de voisins par couche
    batch_size = 20,  # Taille du lot
    shuffle = False
    )

batch = 0
for i in G1_data_loader:
    batch = i
    break

In [986]:
import torch
from torch import nn
import torch.nn.functional as F
import torch
from torch_geometric.data import Data
import random

class ConvE(nn.Module):

    def __init__(self, config):
        super(ConvE, self).__init__()
        self.inp_drop = nn.Dropout(config['input_drop'])
        self.hidden_drop = nn.Dropout(config['hidden_drop'])
        self.feature_map_drop = nn.Dropout2d(config['feat_drop'])
        self.emb_dim1 = config['embedding_shape1']
        self.emb_dim2 = config['embedding_dim'] // self.emb_dim1

        self.conv1 = nn.Conv2d(1, 32, (3, 3), 1, 0, bias=config['use_bias'])
        self.bn0 = nn.BatchNorm2d(1)
        self.bn1 = nn.BatchNorm2d(32)
        self.bn2 = nn.BatchNorm1d(config['embedding_dim'])
        self.fc = nn.Linear(config['hidden_size'], config['embedding_dim'])
        self.register_parameter('b', nn.Parameter(torch.zeros(config['num_entities'])))
        # self.all_entities = torch.stack(list(self.entities_initial_emb.values())).to(config['device'])

   
    def forward(self, e1, rel, e2):
        e1_embedded= e1.view(-1, 1, self.emb_dim1, self.emb_dim2)
        rel_embedded = rel.view(-1, 1, self.emb_dim1, self.emb_dim2)
        stacked_inputs = torch.cat([e1_embedded, rel_embedded], 2)
        x = self.bn0(stacked_inputs)
        x = self.inp_drop(x)
        x = self.conv1(x)
        x = F.relu(self.bn1(x))
        x = self.feature_map_drop(x)
        x = x.view(x.shape[0], -1)
        x = self.fc(x)
        x = self.hidden_drop(x)
        # Check batch size before applying BatchNorm1d
        if x.size(0) > 1:  # Only apply batch norm if batch size > 1
            x = F.relu(self.bn2(x))
        else:
            x = F.relu(x)
        
        # x = F.relu(self.bn2(x))
        # e2_embedded = self.get_embeddings_from_keys(e2, is_entity=True)
        scores = torch.sum(x * e2, dim=1)
        return torch.sigmoid(scores)

In [993]:
R_decoder = ConvE(config).to(config["device"])
data = data.to(config["device"])

In [994]:
score = R_decoder.forward(data.x[0], data.x[1], data.x[1])

RuntimeError: The size of tensor a (256) must match the size of tensor b (768) at non-singleton dimension 1

In [28]:
score.shape

torch.Size([1])

In [32]:
positive_triplets = get_positives(batch)
negative_triplets = generate_negatives(data, batch)
entity_embeddings = batch.x
relation_embeddings = generate_relation_embeddings_tensor(list(gdp.predicate_to_id.values()), 768, "cuda",seed=42)

In [33]:
dl = create_data_loader(positive_triplets, negative_triplets, entity_embeddings, relation_embeddings, 10, shuffle = True)

In [37]:
dl_elt = 0
for i in dl:
    dl_0 = i
    break

In [39]:
s = R_decoder.forward(dl_0[0], dl_0[1].to("cuda"), dl_0[2])
# [  20,   57,    0]

In [41]:
s.shape

torch.Size([10])

[0,
 1,
 2,
 3,
 4,
 5,
 6,
 7,
 8,
 9,
 10,
 11,
 12,
 13,
 14,
 15,
 16,
 17,
 18,
 19,
 20,
 21,
 22,
 23,
 24,
 25,
 26,
 27,
 28,
 29,
 30,
 31,
 32,
 33,
 34,
 35,
 36,
 37,
 38,
 39,
 40,
 41,
 42,
 43,
 44,
 45,
 46,
 47,
 48,
 49,
 50,
 51,
 52,
 53,
 54,
 55,
 56,
 57,
 58,
 59,
 60,
 61,
 62,
 63,
 64,
 65,
 66,
 67,
 68,
 69,
 70,
 71,
 72,
 73,
 74,
 75,
 76,
 77,
 78,
 79,
 80,
 81,
 82,
 83,
 84,
 85,
 86,
 87,
 88,
 89,
 90,
 91,
 92,
 93,
 94,
 95,
 96,
 97,
 98,
 99,
 100,
 101,
 102,
 103,
 104,
 105,
 106,
 107,
 108,
 109]

In [191]:

convE.forward(dl_0[0].to("cuda"), dl_0[1].to("cuda"), dl_0[2].to("cuda"))

RuntimeError: Deterministic behavior was enabled with either `torch.use_deterministic_algorithms(True)` or `at::Context::setDeterministicAlgorithms(true)`, but this operation is not deterministic because it uses CuBLAS and you have CUDA >= 10.2. To enable deterministic behavior in this case, you must set an environment variable before running your PyTorch application: CUBLAS_WORKSPACE_CONFIG=:4096:8 or CUBLAS_WORKSPACE_CONFIG=:16:8. For more information, go to https://docs.nvidia.com/cuda/cublas/index.html#cublasApi_reproducibility

In [149]:
dl_0

[tensor([[-0.3258,  0.0480, -0.3840,  ..., -0.4210, -0.4248,  0.0300],
         [ 0.3105,  0.0069,  0.0499,  ..., -0.0981, -0.1733, -0.0013],
         [-0.0739, -0.0963, -0.2068,  ..., -0.2810, -0.3432,  0.1107],
         ...,
         [ 0.4280, -0.0452, -0.3428,  ..., -0.1099, -0.0211,  0.1333],
         [-0.3806, -0.4151,  0.2006,  ..., -0.2863, -0.5843,  0.1473],
         [-0.7251, -0.1501,  0.0487,  ..., -0.5548, -0.6199,  0.1667]],
        device='cuda:0'),
 tensor([[ 1.3382,  1.1979, -0.5832,  ...,  0.5020, -0.3231,  2.0743],
         [-0.5072,  1.1459,  0.2267,  ..., -0.5854,  0.6170, -1.5064],
         [ 1.3382,  1.1979, -0.5832,  ...,  0.5020, -0.3231,  2.0743],
         ...,
         [ 1.3057,  0.4342, -0.7147,  ..., -0.3842, -0.3943, -0.2614],
         [ 1.3382,  1.1979, -0.5832,  ...,  0.5020, -0.3231,  2.0743],
         [ 1.3382,  1.1979, -0.5832,  ...,  0.5020, -0.3231,  2.0743]],
        grad_fn=<StackBackward0>),
 tensor([[ 0.0371,  0.0778, -0.1921,  ..., -0.3922, -0.26

In [109]:
data.x.shape

torch.Size([52689, 768])

In [62]:
data.edge_index

tensor([[    0,     0,     1,  ..., 52685, 52686, 52688],
        [    1, 32258,   918,  ...,  3997, 37379,  1490]], device='cuda:0')

In [63]:
data.edge_type.

tensor([57, 57, 49,  ..., 57, 57, 57], device='cuda:0')

In [64]:
gdp.predicate_to_id

{'accelerate': 0,
 'acquire': 1,
 'adapt': 2,
 'affect': 3,
 'aim': 4,
 'analyze': 5,
 'arise': 6,
 'associate': 7,
 'avoid': 8,
 'base': 9,
 'become': 10,
 'bring': 11,
 'call': 12,
 'cause': 13,
 'challenge': 14,
 'change': 15,
 'characterize': 16,
 'choose': 17,
 'classifie': 18,
 'combine': 19,
 'come': 20,
 'compare': 21,
 'confirm': 22,
 'consume': 23,
 'contribute': 24,
 'convert': 25,
 'correlate': 26,
 'define': 27,
 'demonstrate': 28,
 'depend': 29,
 'describe': 30,
 'discuss': 31,
 'display': 32,
 'distinguish': 33,
 'divide': 34,
 'eliminate': 35,
 'ensure': 36,
 'execute': 37,
 'exist': 38,
 'explore': 39,
 'extract': 40,
 'face': 41,
 'feature': 42,
 'focus': 43,
 'follow': 44,
 'guarantee': 45,
 'guide': 46,
 'highlight': 47,
 'identify': 48,
 'illustrate': 49,
 'impose': 50,
 'improve': 51,
 'include': 52,
 'incorporate': 53,
 'integrate': 54,
 'interact': 55,
 'introduce': 56,
 'is': 57,
 'know': 58,
 'lack': 59,
 'learn': 60,
 'limit': 61,
 'link': 62,
 'manage': 63,


In [30]:
relations = u.read_pickle_file(Edges)

In [29]:
def create_triplet_lookup(data):
    triplet_set = set()
    for i in range(data.edge_index.size(1)):
        src = data.edge_index[0, i].item()
        dest = data.edge_index[1, i].item()
        relation = data.edge_type[i].item()
        triplet_set.add((src, relation, dest))
    return triplet_set


def is_triplet_in_data(triplet_set, triplet):
    return triplet in triplet_set

def generate_negatives(data, batch, negative_ratio=1, relation_weight=None):
    """
    Generate negative triplets dynamically for a given batch of triplets with validation 
    and resampling for existing triplets in the graph.
    """
    edge_index = batch.edge_index  # (2, num_edges)
    edge_type = batch.edge_type    # (num_edges,)
    num_nodes = batch.num_nodes
    num_relations = batch.num_relations if hasattr(batch, "num_relations") else torch.max(edge_type).item() + 1

    positives = []
    negatives = []

    # Precompute triplet lookup
    triplet_set = create_triplet_lookup(data)

    # Collect all positive triplets
    for i in range(edge_index.size(1)):
        h, t = edge_index[:, i]
        r = edge_type[i]
        positives.append((h.item(), r.item(), t.item()))

    # Generate negatives
    for h, r, t in positives:
        for _ in range(negative_ratio):
            while True:  # Keep sampling until a valid negative is found
                if random.random() < 0.33:
                    # Corrupt head
                    h_neg = random.randint(0, num_nodes - 1)
                    global_h_neg = batch.n_id[h_neg] if hasattr(batch, 'n_id') else h_neg
                    if not is_triplet_in_data(triplet_set, (global_h_neg, r, batch.n_id[t])):
                        negatives.append((h_neg, r, t))
                        break  # Valid negative found, exit loop
                elif random.random() < 0.66:
                    # Corrupt tail
                    t_neg = random.randint(0, num_nodes - 1)
                    global_t_neg = batch.n_id[t_neg] if hasattr(batch, 'n_id') else t_neg
                    if not is_triplet_in_data(triplet_set, (batch.n_id[h], r, global_t_neg)):
                        negatives.append((h, r, t_neg))
                        break  # Valid negative found, exit loop
                else:
                    # Corrupt relation
                    if relation_weight:
                        r_neg = random.choices(
                            list(relation_weight.keys()),
                            weights=list(relation_weight.values()),
                            k=1
                        )[0]
                    else:
                        r_neg = random.randint(0, num_relations - 1)
                    if not is_triplet_in_data(triplet_set, (batch.n_id[h], r_neg, batch.n_id[t])):
                        negatives.append((h, r_neg, t))
                        break  # Valid negative found, exit loop

    # Convert negatives to tensor
    negative_tensor = torch.tensor(negatives, dtype=torch.long)
    return negative_tensor

def get_positives(batch):
    """
    Generate a tensor of all positive triplets (head, relation, tail) from the given batch.
    """
    edge_index = batch.edge_index  # (2, num_edges)
    edge_type = batch.edge_type    # (num_edges,)

    positives = []

    # Collect all positive triplets
    for i in range(edge_index.size(1)):
        h, t = edge_index[:, i]
        r = edge_type[i]
        positives.append((h.item(), r.item(), t.item()))

    # Convert positives to tensor
    positive_tensor = torch.tensor(positives, dtype=torch.long)
    return positive_tensor

In [46]:
data

Data(x=[52689, 768], edge_index=[2, 65155], edge_attr=[65155, 768], edge_type=[65155])

In [4]:
import numpy as np
import random
import torch
def set_seed(seed):
    """
    Set the random seed for reproducibility.

    Args:
        seed (int): The random seed to use.
    """
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    # if torch.cuda.is_available():
    #     torch.cuda.manual_seed(seed)
    #     torch.cuda.manual_seed_all(seed)  # For multi-GPU setups


set_seed(42)

In [834]:
data

Data(x=[52689, 768], edge_index=[2, 65155], edge_attr=[65155, 768], edge_type=[65155])

In [76]:
def count_valid_elements(batch, removed):
    """
    Compte le nombre d'arêtes dans batch.e_id qui sont dans `removed` et dont
    le nœud de destination est présent dans batch.input_id.

    Args:
        batch (dict): Contient les clés 'e_id', 'edge_index', et 'input_id'.
                      - e_id: Tensor des identifiants des arêtes.
                      - edge_index: Tensor [2, num_edges] représentant les arêtes du graphe.
                      - input_id: Tensor des nœuds cibles du batch.
        removed (Tensor): Tensor des indices d'arêtes à vérifier.

    Returns:
        int: Nombre d'éléments qui satisfont la condition.
    """
    # Trouver les indices des arêtes dans removed
    mask = torch.isin(batch["e_id"], removed)
    indices = torch.nonzero(mask, as_tuple=True)[0]

    # Obtenir les nœuds de destination des arêtes correspondantes
    dest_nodes = batch["edge_index"][1][indices]

    # Vérifier si ces nœuds de destination sont dans batch.input_id
    is_in_input_id = torch.isin(dest_nodes, batch["input_id"])

    # Retourner le nombre d'éléments qui satisfont la condition
    return is_in_input_id.sum().item()


from torch_geometric.loader import NeighborLoader

# Liste des nœuds cibles
target_nodes = torch.tensor([55, 60])

# Création du DataLoader
set_seed(42)
loader = NeighborLoader(
    data,
    input_nodes=target_nodes,  # Les nœuds que tu veux embeder
    num_neighbors=[3, 10],  # Nombre de voisins à échantillonner par couche
    batch_size=2,
    shuffle=False
)
# removed = torch.tensor([99,97])
# # Obtenir un batch contenant les nœuds et leurs voisins
# for batch in loader:
#     print(batch.n_id[2])
    
#     print(count_valid_elements(batch,removed))

In [77]:
for i in loader:
    print(i)

Data(x=[4, 768], edge_index=[2, 2], edge_attr=[2, 768], edge_type=[2], n_id=[4], e_id=[2], input_id=[2], batch_size=2)


In [22]:
removed = torch.tensor([99])


# Vérifiez si les éléments de tensor_1 sont dans tensor_2
mask = torch.isin(tensor_1, tensor_2)

# Obtenez les indices des éléments correspondants dans tensor_1
indices = torch.nonzero(mask, as_tuple=True)[0]

NameError: name 'tensor_1' is not defined

In [62]:
data

Data(x=[52689, 768], edge_index=[2, 65155], edge_attr=[65155, 768], edge_type=[65155])

In [71]:
for i in G1_data_loader:
    print(i)
    break

Data(x=[863, 768], edge_index=[2, 870], edge_attr=[870, 768], edge_type=[870], n_id=[863], e_id=[870], input_id=[20], batch_size=20)


In [78]:
import torch

# Exemple de deux tensors
tensor1 = torch.tensor([1, 2, 3, 4, 5])
tensor2 = torch.tensor([  1, 6, 7])

# Trouver l'intersection
intersection = tensor1[torch.isin(tensor1, tensor2)]

if len(intersection):
    print(intersection) 

tensor([1])


In [100]:
import random

# Définition des types de tuyaux et leurs connexions
TYPES = {
    "X": [],  # Vide
    "C": ["N", "E", "S", "W"],  # Croix (toutes les directions ouvertes)
    "L": ["N", "E"],  # Coin (Nord-Est)
    "I": ["N", "S"],  # Ligne (Nord-Sud)
    "T": ["N", "E", "W"],  # T (Nord, Est, Ouest)
    "V": ["N"],  # Simple (Nord ouvert)
}

ROTATIONS = {
    "N": 0,
    "E": 90,
    "S": 180,
    "W": 270
}

def rotate_directions(directions, angle):
    """Tourner les directions en fonction de l'angle donné."""
    order = ["N", "E", "S", "W"]
    return [order[(order.index(d) + angle // 90) % 4] for d in directions]

def generate_grid(n, m):
    """Génère une grille nxm avec une solution garantie."""
    grid = [[None for _ in range(m)] for _ in range(n)]
    
    # Générer une solution en parcourant la grille
    for i in range(n):
        for j in range(m):
            if i == 0 and j == 0:
                # Point d'entrée
                tile_type = "I"
                orientation = 90  # Horizontal
            elif i == n-1 and j == m-1:
                # Point de sortie
                tile_type = "I"
                orientation = 90  # Horizontal
            else:
                # Type aléatoire de tuyau
                tile_type = random.choice(list(TYPES.keys()))
                orientation = random.choice(list(ROTATIONS.values()))
            
            # Appliquer la rotation
            directions = rotate_directions(TYPES[tile_type], orientation)
            
            # Ajouter à la grille
            grid[i][j] = (tile_type, directions)
    
    # Vérifier la validité et ajuster si nécessaire
    grid = validate_grid(grid, n, m)
    return grid

def validate_grid(grid, n, m):
    """Valide et ajuste la grille pour assurer une solution."""
    for i in range(n):
        for j in range(m):
            if i > 0:  # Vérifier le haut
                if "S" in grid[i-1][j][1] and "N" not in grid[i][j][1]:
                    grid[i][j] = ("I", ["N", "S"])
            if j > 0:  # Vérifier la gauche
                if "E" in grid[i][j-1][1] and "W" not in grid[i][j][1]:
                    grid[i][j] = ("I", ["W", "E"])
    return grid

# Affichage simple
def display_grid_improved(grid):
    """Affiche la grille avec des symboles pour les tuyaux et leurs orientations."""
    # Mapping des types de tuyaux et directions vers des symboles visuels
    SYMBOLS = {
        "X": " ",  # Case vide
        "C": "╬",  # Croix (toutes les directions ouvertes)
        "L": "╚",  # Coin
        "I": "║",  # Ligne verticale
        "T": "╦",  # T
        "V": "╥",  # Simple
    }
    
    ROTATED_SYMBOLS = {
        ("I", 0): "║", ("I", 90): "═", ("I", 180): "║", ("I", 270): "═",
        ("L", 0): "╚", ("L", 90): "╔", ("L", 180): "╗", ("L", 270): "╝",
        ("T", 0): "╦", ("T", 90): "╣", ("T", 180): "╩", ("T", 270): "╠",
        ("C", 0): "╬", ("C", 90): "╬", ("C", 180): "╬", ("C", 270): "╬",
        ("V", 0): "╥", ("V", 90): "╡", ("V", 180): "╨", ("V", 270): "╞",
    }
    
    print("\nGenerated Grid:")
    for row in grid:
        row_display = []
        for tile in row:
            tile_type, directions = tile
            orientation = ROTATIONS[directions[0]] if directions else 0
            symbol = ROTATED_SYMBOLS.get((tile_type, orientation), SYMBOLS.get(tile_type, "?"))
            row_display.append(symbol)
        print(" ".join(row_display))
    print()

In [460]:
def solve_grid(grid):
    """Résout la grille en ajustant les tuyaux pour assurer une solution."""
    n, m = len(grid), len(grid[0])  # Dimensions de la grille

    # Vérification des connexions adjacentes
    def is_safe(x, y, dx, dy):
        """Vérifie si deux cellules adjacentes sont connectées de manière sûre."""
        if 0 <= x + dx < n and 0 <= y + dy < m:
            current_directions = grid[x][y][1]
            neighbor_directions = grid[x + dx][y + dy][1]
            
            if "N" in current_directions and "S" in neighbor_directions and dx == -1:
                return True
            if "S" in current_directions and "N" in neighbor_directions and dx == 1:
                return True
            if "W" in current_directions and "E" in neighbor_directions and dy == -1:
                return True
            if "E" in current_directions and "W" in neighbor_directions and dy == 1:
                return True
        return False

    def rotate_pipe(pipe, orientation):
        """Tourne un tuyau à une orientation donnée."""
        directions = TYPES[pipe]
        return rotate_directions(directions, orientation)

    def adjust_cell(x, y):
        """Ajuste une cellule pour qu'elle se connecte correctement à ses voisins."""
        if x == 0 and y == 0:  # Point d'entrée
            grid[x][y] = ("I", ["E"])  # Orientation vers l'Est
            return True
        if x == n - 1 and y == m - 1:  # Point de sortie
            grid[x][y] = ("I", ["W"])  # Orientation vers l'Ouest
            return True

        # Essayer toutes les rotations pour les autres cellules
        for orientation in [0, 90, 180, 270]:
            grid[x][y] = (grid[x][y][0], rotate_pipe(grid[x][y][0], orientation))
            safe = True
            for dx, dy in [(-1, 0), (1, 0), (0, -1), (0, 1)]:
                if not is_safe(x, y, dx, dy):
                    safe = False
                    break
            if safe:
                return True
        return False

    # Ajuster les cellules clé (proches des points d'entrée et de sortie)
    def adjust_key_cells():
        """Ajuste les cellules proches des points d'entrée et de sortie."""
        if not adjust_cell(0, 1):  # Cellule juste après le point d'entrée
            grid[0][1] = ("I", ["W", "E"])  # Ligne droite horizontale
        if not adjust_cell(n - 1, m - 2):  # Cellule juste avant le point de sortie
            grid[n - 1][m - 2] = ("I", ["W", "E"])  # Ligne droite horizontale

    # Ajuster les cellules clé
    adjust_key_cells()

    # Parcours de la grille pour ajuster les autres tuyaux
    for i in range(n):
        for j in range(m):
            if not adjust_cell(i, j):
                print(f"Impossible d'ajuster la cellule ({i}, {j}).")
                return None

    return grid

# Exemple d'utilisation
n, m = 5, 5  # Taille de la grille
grid = generate_grid(n, m)  # Générer une grille aléatoire
print("Grille générée :")
display_grid_improved(grid)  # Afficher la grille initiale

solution = solve_grid(grid)  # Résoudre la grille
if solution:
    print("Grille résolue :")
    display_grid_improved(solution)


Grille générée :

Generated Grid:
═ ═ ═ ═ ╬
╥ ╥ ═ ═ ╬
║ ╨ ╣ ═ ═
╬ ╬ ═ ═ ╩
║ ╦ ╞ ╝ ║

Impossible d'ajuster la cellule (0, 1).


In [467]:
import random

# Types de tuyaux et leurs connexions
TYPES = {
    "X": [],  # Vide
    "I": ["N", "S"],  # Ligne verticale
    "L": ["N", "E"],  # Coin
    "T": ["N", "E", "W"],  # T
    "C": ["N", "E", "S", "W"],  # Croix
}

def get_opposite_direction(direction):
    """Renvoie la direction opposée."""
    opposites = {"N": "S", "S": "N", "E": "W", "W": "E"}
    return opposites[direction]

def generate_valid_grid(n, m):
    """Génère une grille valide avec un chemin connecté de l'entrée à la sortie."""
    grid = [[None for _ in range(m)] for _ in range(n)]

    # Étape 1 : Créer un chemin valide entre (0, 0) et (n-1, m-1)
    x, y = 0, 0
    while x < n - 1 or y < m - 1:
        if x < n - 1 and (y == m - 1 or random.choice([True, False])):
            # Descendre
            grid[x][y] = ("I", ["N", "S"])
            x += 1
        elif y < m - 1:
            # Aller à droite
            grid[x][y] = ("I", ["W", "E"])
            y += 1

    # Dernière cellule (sortie)
    grid[x][y] = ("I", ["N", "S"])

    # Étape 2 : Compléter la grille avec des tuyaux aléatoires
    for i in range(n):
        for j in range(m):
            if grid[i][j] is None:
                pipe_type = random.choice(list(TYPES.keys()))
                directions = TYPES[pipe_type]
                grid[i][j] = (pipe_type, directions)

    return grid

def display_grid(grid):
    """Affiche la grille générée."""
    SYMBOLS = {
        "X": " ",  # Vide
        "I": "║",  # Ligne verticale
        "L": "╚",  # Coin
        "T": "╦",  # T
        "C": "╬",  # Croix
    }
    for row in grid:
        print(" ".join(SYMBOLS[tile[0]] if tile else "?" for tile in row))
    print()

# Exemple d'utilisation
n, m = 5, 5  # Taille de la grille
grid = generate_valid_grid(n, m)
print("Grille générée (valide) :")
display_grid(grid)


Grille générée (valide) :
║ ║ ║ ║  
║ ╦ ║ ║ ╦
║ ╬ ╬ ║ ║
║   ╚ ╚ ║
╬ ╚   ╚ ║



In [519]:
import random

# Définition des types de tuyaux et leurs connexions
PIPE_TYPES = {
    "empty": {"openings": [], "symbol": " "},  # Case vide
    "line": {"openings": ["N", "S"], "symbol": "║"},
    "curve": {"openings": ["N", "E"], "symbol": "╚"},
    "cross": {"openings": ["N", "S", "E", "W"], "symbol": "╬"},
    "tee": {"openings": ["N", "S", "E"], "symbol": "╦"},
}

DIRECTIONS = ["N", "E", "S", "W"]
DXDY = {"N": (-1, 0), "E": (0, 1), "S": (1, 0), "W": (0, -1)}  # Déplacement par direction


def rotate_openings(openings, rotation):
    """Fait pivoter les ouvertures d'un tuyau."""
    index_shift = rotation // 90
    return [DIRECTIONS[(DIRECTIONS.index(dir) + index_shift) % 4] for dir in openings]


def rotate_symbol(symbol, rotation):
    """Fait pivoter les symboles."""
    return symbol  # Simplifié ici, les symboles n'ont pas besoin de changer pour cette implémentation


def is_inside_board(x, y, rows, cols):
    """Vérifie si une position est à l'intérieur de la grille."""
    return 1 <= x <= rows and 1 <= y <= cols


def find_path(board, x, y, target_x, target_y, rows, cols, visited):
    """Algorithme récursif pour trouver un chemin entre A et B."""
    if (x, y) == (target_x, target_y):
        return [(x, y)]  # On a atteint la cible

    visited.add((x, y))

    for direction, (dx, dy) in DXDY.items():
        next_x, next_y = x + dx, y + dy

        if is_inside_board(next_x, next_y, rows, cols) and (next_x, next_y) not in visited:
            path = find_path(board, next_x, next_y, target_x, target_y, rows, cols, visited)
            if path:  # Si un chemin valide est trouvé
                return [(x, y)] + path

    visited.remove((x, y))  # Backtracking
    return None


def generate_valid_board(rows, cols):
    """Génère une grille avec une solution valide et sans tuyaux ouverts exposés."""
    board = [[None for _ in range(cols + 2)] for _ in range(rows + 2)]

    # Ajouter des bordures fermées
    for i in range(rows + 2):
        board[i][0] = "border"  # Bord gauche
        board[i][cols + 1] = "border"  # Bord droit
    for j in range(cols + 2):
        board[0][j] = "border"  # Bord haut
        board[rows + 1][j] = "border"  # Bord bas

    # Définir le point d'entrée (A) et de sortie (B)
    board[1][0] = {"type": "line", "rotation": 0, "symbol": "║"}  # Point d'entrée
    board[rows][cols + 1] = {"type": "line", "rotation": 180, "symbol": "║"}  # Point de sortie

    # Trouver un chemin valide entre A et B
    path = find_path(board, 1, 1, rows, cols, rows, cols, set())
    if not path:
        raise ValueError("Impossible de générer un chemin valide.")

    # Remplir le chemin avec des tuyaux connectés
    for i, (x, y) in enumerate(path):
        if i == 0:  # Point d'entrée
            continue
        if i == len(path) - 1:  # Point de sortie
            continue
        prev_x, prev_y = path[i - 1]
        next_x, next_y = path[i + 1]
        openings = []
        if (x - prev_x, y - prev_y) in DXDY.values():
            openings.append(DIRECTIONS[list(DXDY.values()).index((x - prev_x, y - prev_y))])
        if (next_x - x, next_y - y) in DXDY.values():
            openings.append(DIRECTIONS[list(DXDY.values()).index((next_x - x, next_y - y))])
        # Trouver un type de tuyau correspondant ou utiliser un type par défaut
        pipe_type = next(
            (pipe for pipe, details in PIPE_TYPES.items() if set(details["openings"]) == set(openings)),
            "cross"  # Par défaut : "cross"
        )
        board[x][y] = {
            "type": pipe_type,
            "rotation": 0,
            "symbol": PIPE_TYPES[pipe_type]["symbol"],
        }

    # Remplir les cases restantes avec des tuyaux ou des cases vides
    for i in range(1, rows + 1):
        for j in range(1, cols + 1):
            if board[i][j] is None:
                pipe_type = random.choice(list(PIPE_TYPES.keys()))
                rotation = random.choice([0, 90, 180, 270])
                openings = rotate_openings(PIPE_TYPES[pipe_type]["openings"], rotation)
                symbol = rotate_symbol(PIPE_TYPES[pipe_type]["symbol"], rotation)
                board[i][j] = {
                    "type": pipe_type,
                    "rotation": rotation,
                    "openings": openings,
                    "symbol": symbol,
                }

    return board


def print_board(board):
    """Affiche la grille."""
    for row in board:
        row_str = []
        for cell in row:
            if cell == "border":
                row_str.append("█")
            elif cell is None:
                row_str.append(" ")
            elif isinstance(cell, dict):
                row_str.append(cell["symbol"])
            else:
                row_str.append(" ")
        print("".join(row_str))


# Exemple d'utilisation
rows, cols = 10, 10
board = generate_valid_board(rows, cols)
print_board(board)


████████████
║╦╬╬╬╬╬╬╬╬╬█
█╚╬╦╚ ╦╚╬║╬█
█╬ ╬╚ ╬   ╬█
█╚║║╬  ╦╦ ╬█
█╦╬ ║╚╦╦╬ ╬█
█╦ ║╦ ╦╦║╚╬█
█ ╦║║ ╦╦╦║╬█
█║╚╚╬║╦ ╚ ╬█
█   ║║╬╚ ╚╬█
█ ╦╬ ╚║ ╦╬╬║
████████████


In [524]:
def is_connection_valid(board, x1, y1, x2, y2):
    """
    Vérifie si deux cellules sont connectées correctement.
    """
    if board[x1][y1] is None or board[x2][y2] is None:
        return False

    # Vérifier que les deux cellules contiennent des tuyaux valides
    if "openings" not in board[x1][y1] or "openings" not in board[x2][y2]:
        return False

    # Récupérer les ouvertures des deux tuyaux
    openings1 = set(board[x1][y1]["openings"])
    openings2 = set(board[x2][y2]["openings"])

    # Calculer la direction entre les deux cellules
    if x2 == x1 - 1 and y2 == y1:  # Nord
        return "N" in openings1 and "S" in openings2
    if x2 == x1 + 1 and y2 == y1:  # Sud
        return "S" in openings1 and "N" in openings2
    if y2 == y1 + 1 and x2 == x1:  # Est
        return "E" in openings1 and "W" in openings2
    if y2 == y1 - 1 and x2 == x1:  # Ouest
        return "W" in openings1 and "E" in openings2

    return False


def solve_puzzle(board, x, y, target_x, target_y, visited=None):
    """
    Résout le puzzle en trouvant une solution connectée de (x, y) à (target_x, target_y).
    Utilise une recherche en profondeur (DFS).
    """
    if visited is None:
        visited = set()

    # Si nous atteignons la cellule cible, la solution est trouvée
    if (x, y) == (target_x, target_y):
        return [(x, y)]

    visited.add((x, y))

    # Essayer toutes les directions possibles
    for direction, (dx, dy) in DXDY.items():
        next_x, next_y = x + dx, y + dy

        # Vérifier que la cellule voisine est valide et connectée
        if (
            is_inside_board(next_x, next_y, len(board) - 2, len(board[0]) - 2)
            and (next_x, next_y) not in visited
            and is_connection_valid(board, x, y, next_x, next_y)
        ):
            # Continuer la recherche depuis la cellule voisine
            path = solve_puzzle(board, next_x, next_y, target_x, target_y, visited)
            if path:  # Si un chemin valide est trouvé
                return [(x, y)] + path

    visited.remove((x, y))  # Backtracking
    return None


def visualize_solution(board, solution):
    """
    Met à jour la grille pour afficher le chemin de la solution avec des symboles spéciaux.
    """
    for x, y in solution:
        if board[x][y] is not None and board[x][y] != "border":
            board[x][y]["symbol"] = "●"  # Utiliser un symbole pour marquer le chemin



Grille générée :
███████
║╚╬╬╬╬█
█╚╬╦╚╬█
█╬ ║║╬█
█║╚ ╦╬█
█╦╦╚╬╚║
███████

Aucune solution trouvée.


In [892]:

# Exemple d'utilisation
rows, cols = 5, 5
board = generate_valid_board(rows, cols)

print("Grille générée :")
print_board(board)

# Résoudre le puzzle
start_x, start_y = 1, 1
target_x, target_y = rows, cols
solution = solve_puzzle(board, start_x, start_y, target_x, target_y)

if solution:
    print("\nSolution trouvée :")
    visualize_solution(board, solution)
    print_board(board)
else:
    print("\nAucune solution trouvée.")


Grille générée :
███████
║╚╬╬╬╬█
█╬╚╬╬╬█
█╬╬╚╦╬█
█╦╬╬╦╬█
█ ║╚║║║
███████

Aucune solution trouvée.


In [894]:
import random
import itertools

# Définition des types de tuyaux et leurs connexions
PIPE_TYPES = {
    "empty": {"openings": [], "symbol": " "},  # Case vide
    "line": {"openings": ["N", "S"], "symbol": "║"},
    "curve": {"openings": ["N", "E"], "symbol": "╚"},
    "cross": {"openings": ["N", "S", "E", "W"], "symbol": "╬"},
    "tee": {"openings": ["N", "S", "E"], "symbol": "╦"},
}

DIRECTIONS = ["N", "E", "S", "W"]
DXDY = {"N": (-1, 0), "E": (0, 1), "S": (1, 0), "W": (0, -1)}  # Déplacement par direction
REVERSE_DIRECTION = {"N": "S", "S": "N", "E": "W", "W": "E"}

def rotate_pipe(pipe_type, rotation):
    """Fait pivoter un tuyau et ses ouvertures."""
    if rotation == 0:
        return pipe_type
    
    openings = PIPE_TYPES[pipe_type]["openings"]
    rotated_openings = [DIRECTIONS[(DIRECTIONS.index(dir) + rotation // 90) % 4] for dir in openings]
    
    # Trouver le type de tuyau correspondant aux nouvelles ouvertures
    for type_name, type_info in PIPE_TYPES.items():
        if set(type_info["openings"]) == set(rotated_openings):
            return type_name
    
    return pipe_type

def generate_valid_board(rows, cols, max_attempts=100):
    """Génère une grille avec une solution garantie."""
    for _ in range(max_attempts):
        board = [[None for _ in range(cols + 2)] for _ in range(rows + 2)]
        
        # Ajouter des bordures fermées
        for i in range(rows + 2):
            board[i][0] = "border"
            board[i][cols + 1] = "border"
        for j in range(cols + 2):
            board[0][j] = "border"
            board[rows + 1][j] = "border"
        
        # Configurer les points d'entrée et de sortie
        board[1][0] = {"type": "line", "rotation": 0, "symbol": "║"}
        board[rows][cols + 1] = {"type": "line", "rotation": 180, "symbol": "║"}
        
        # Remplir le reste de la grille
        for x in range(1, rows + 1):
            for y in range(1, cols + 1):
                if board[x][y] is None:
                    pipe_type = random.choice(list(PIPE_TYPES.keys()))
                    rotation = random.choice([0, 90, 180, 270])
                    board[x][y] = {
                        "type": pipe_type,
                        "rotation": rotation,
                        "symbol": PIPE_TYPES[pipe_type]["symbol"]
                    }
        
        # Vérifier s'il existe une solution
        solution = find_solution(board, rows, cols)
        if solution:
            return board, solution
    
    raise ValueError("Impossible de générer une grille avec une solution après plusieurs tentatives.")

def find_solution(board, rows, cols):
    """Trouve une solution pour la grille de tuyaux."""
    def is_connected(current, next_cell, direction):
        """Vérifie si deux cellules sont connectées dans une direction donnée."""
        # Obtenir les ouvertures des tuyaux
        current_type = current['type']
        next_type = next_cell['type']
        
        # Vérifier si la direction est dans les ouvertures
        return (direction in PIPE_TYPES[current_type]['openings'] and 
                REVERSE_DIRECTION[direction] in PIPE_TYPES[next_type]['openings'])
    
    def dfs(x, y, visited, path):
        """Recherche en profondeur pour trouver un chemin."""
        if x == rows and y == cols:
            return path
        
        visited.add((x, y))
        
        # Essayer toutes les directions
        for direction, (dx, dy) in DXDY.items():
            next_x, next_y = x + dx, y + dy
            
            # Vérifier les limites et que la case n'a pas été visitée
            if (1 <= next_x <= rows and 1 <= next_y <= cols and 
                (next_x, next_y) not in visited):
                current_cell = board[x][y]
                next_cell = board[next_x][next_y]
                
                # Vérifier si les tuyaux sont connectés
                if is_connected(current_cell, next_cell, direction):
                    result = dfs(next_x, next_y, visited, path + [(next_x, next_y)])
                    if result:
                        return result
        
        visited.remove((x, y))
        return None
    
    # Commencer la recherche du point de départ
    solution = dfs(1, 1, set(), [(1, 1)])
    return solution

def print_board(board):
    """Affiche la grille avec des symboles."""
    for row in board:
        row_str = []
        for cell in row:
            if cell == "border":
                row_str.append("█")
            elif cell is None:
                row_str.append(" ")
            elif isinstance(cell, dict):
                row_str.append(cell["symbol"])
            else:
                row_str.append(" ")
        print("".join(row_str))

def print_solution(board, solution):
    """Affiche la grille avec le chemin de la solution surlignée."""
    solution_set = set(solution)
    for x in range(1, len(board) - 1):
        row_str = []
        for y in range(1, len(board[x]) - 1):
            if (x, y) in solution_set:
                # Surligner les cases de la solution
                row_str.append("\033[92m" + board[x][y]["symbol"] + "\033[0m")
            elif board[x][y] == "border":
                row_str.append("█")
            elif board[x][y] is None:
                row_str.append(" ")
            elif isinstance(board[x][y], dict):
                row_str.append(board[x][y]["symbol"])
            else:
                row_str.append(" ")
        print("".join(row_str))

def main():
    rows, cols = 10, 10
    while True:
        try:
            # Générer une grille valide
            board, solution = generate_valid_board(rows, cols)
            
            # Afficher la grille générée
            print("Grille générée :")
            print_board(board)
            print("\nSolution :")
            print_solution(board, solution)
            print(f"\nChemin de solution : {solution}")
            
            # Sortir de la boucle si une solution est trouvée
            break
        except ValueError as e:
            print("Tentative échouée, régénération de la grille...\n")


Impossible de générer une grille avec une solution après plusieurs tentatives.


In [977]:
if __name__ == "__main__":
    main()

Tentative échouée, régénération de la grille...

Tentative échouée, régénération de la grille...

Tentative échouée, régénération de la grille...

Tentative échouée, régénération de la grille...

Tentative échouée, régénération de la grille...

Tentative échouée, régénération de la grille...

Tentative échouée, régénération de la grille...

Tentative échouée, régénération de la grille...

Tentative échouée, régénération de la grille...

Tentative échouée, régénération de la grille...

Tentative échouée, régénération de la grille...

Tentative échouée, régénération de la grille...

Tentative échouée, régénération de la grille...

Tentative échouée, régénération de la grille...

Tentative échouée, régénération de la grille...

Tentative échouée, régénération de la grille...

Grille générée :
████████████
║╦╦╦ ║╚ ╬╦║█
█╬║║ ╚║║║╦ █
█║ ╚╬╚╚║╚╬╦█
█╚╬╬╚╚╬║ ╚╚█
█ ║╚╬╚╦║║╚║█
█║ ╦║╦╬║  ╦█
█╬╬╚╚╬║║║ ╚█
█╦  ╦║╚╬╬╬╚█
█  ║║╚║╚║╚╬█
█║╦╬╚║ ║╦║╬║
████████████

Solution :
╦╦╦ ║╚ ╬╦║
╬║║ ╚║║║╦ 
║ ╚╬╚╚║╚╬